- Esse notebook servirá pra observar com mais detalhe oque pode ser extraído dessa API da NFL.

In [1]:
import pandas as pd
import nflreadpy as nfl

# configuração do escopo temporal
ANO_ATUAL = 2026
ultimos_5_anos = list(range(ANO_ATUAL - 5, ANO_ATUAL))

print(f"Extraindo dados para as temporadas: {ultimos_5_anos}")

# retorna placares, times, semanas e, crucialmente, as linhas de aposta (spread/total).
# útil para: criar um 'prior' de expectativa (A zebra esperada pelas casas de aposta).
df_jogos = nfl.load_schedules(ultimos_5_anos).to_pandas()

# retorna estatísticas agregadas por jogador (passes, corridas, recepções, fumbles).
# útil para: calcular Carries (Controle de Relógio) e Turnovers brutos.
df_player_stats = nfl.load_player_stats(ultimos_5_anos).to_pandas()

# PLAY-BY-PLAY (PBP)
# registra cada jogada individual da partida (passes, corridas, punts, penalidades).
# inclui métricas avançadas matemáticas como EPA (Expected Points Added) e WPA (Win Probability Added).
# útil para: 
# 1. isolar o Garbage Time (ex: filtrar jogadas onde a Win Probability do time > 90% ou < 10%).
# 2. criar taxas reais de pressão (Sacks por Dropback, e não apenas Sacks totais).
df_pbp = nfl.load_pbp(ultimos_5_anos).to_pandas()

# RELATÓRIOS DE LESÕES (Injuries)
# registra o status de lesão semana a semana dos jogadores (Questionable, Out, IR).
# útil para: mapear o colapso estrutural. Se um time da "Elite Aérea" perde seu QB titular, 
# o modelo precisa saber que a identidade daquele cluster foi temporariamente desativada.
df_lesoes = nfl.load_injuries(ultimos_5_anos).to_pandas()

# SNAP COUNTS (Contagem de Snaps)
# mostra quantas jogadas cada jogador esteve em campo (Offense, Defense, Special Teams).
# útil para: medir a exaustão defensiva. O "Cluster 5 - Amarelo" da análise indicou que 
# defesas que ficam muito tempo em campo falham. O Snap Count mapeia isso.
df_snaps = nfl.load_snap_counts(ultimos_5_anos).to_pandas()

# ROSTERS (Elencos e Posições)
# traz informações demográficas, posições e status dos jogadores.
# util para: Cruzar com o PBP ou Player Stats e agregar a força do time por unidades específicas 
# (ex: medir a eficiência apenas da Linha Ofensiva ou apenas da Secundária).
df_rosters = nfl.load_rosters(ultimos_5_anos).to_pandas()

# PARTICIPAÇÃO (Personnel/Formações) - *Dependendo da versão/disponibilidade da API*
# registra a combinação de jogadores em campo (ex: 11 personnel - 1 RB, 1 TE, 3 WR).
# ítil para: Identificar times previsíveis. Times do "Cluster Roxo - Rolo Compressor Desdentado"
# costumam usar formações pesadas obvias que a defesa moderna já sabe anular.
try:
    df_participacao = nfl.load_participation(ultimos_5_anos).to_pandas()
    print("Base de Participação carregada com sucesso.")
except AttributeError:
    print("Base de Participação não disponível diretamente nesta versão da API.")

Extraindo dados para as temporadas: [2021, 2022, 2023, 2024, 2025]
Base de Participação carregada com sucesso.


- visualização das colunas de cada base:

In [2]:
# dicionário com todas as bases recém-ingeridas
bases_ingeridas = {
    "Schedules (Jogos)": df_jogos,
    "Player Stats": df_player_stats,
    "Play-by-Play (PBP)": df_pbp,
    "Lesões (Injuries)": df_lesoes,
    "Snap Counts": df_snaps,
    "Rosters (Elencos)": df_rosters
}

for nome, df in bases_ingeridas.items():
    print(f"\n{'='*50}")
    print(f"🏈 {nome} | Formato: {df.shape[0]} linhas x {df.shape[1]} colunas")
    print(f"{'='*50}")
    
    colunas = df.columns.tolist()
    
    # imprime as colunas em blocos de 5 para facilitar a varredura visual
    for i in range(0, len(colunas), 5):
        print("  |  ".join(colunas[i:i+5]))
        
    print("\n")


🏈 Schedules (Jogos) | Formato: 1424 linhas x 46 colunas
game_id  |  season  |  game_type  |  week  |  gameday
weekday  |  gametime  |  away_team  |  away_score  |  home_team
home_score  |  location  |  result  |  total  |  overtime
old_game_id  |  gsis  |  nfl_detail_id  |  pfr  |  pff
espn  |  ftn  |  away_rest  |  home_rest  |  away_moneyline
home_moneyline  |  spread_line  |  away_spread_odds  |  home_spread_odds  |  total_line
under_odds  |  over_odds  |  div_game  |  roof  |  surface
temp  |  wind  |  away_qb_id  |  home_qb_id  |  away_qb_name
home_qb_name  |  away_coach  |  home_coach  |  referee  |  stadium_id
stadium



🏈 Player Stats | Formato: 94848 linhas x 150 colunas
player_id  |  player_name  |  player_display_name  |  position  |  position_group
headshot_url  |  season  |  week  |  season_type  |  game_id
team  |  opponent_team  |  completions  |  attempts  |  passing_yards
passing_tds  |  passing_interceptions  |  sacks_suffered  |  sack_yards_lost  |  sack_fumbles
sac

In [5]:
# salvando os dados brutos

print("Exportando bases brutas para a próxima camada...")
df_jogos.to_parquet('./raw_data/raw_schedules.parquet', index=False)
df_player_stats.to_parquet('./raw_data/raw_player_stats.parquet', index=False)
df_pbp.to_parquet('./raw_data/raw_pbp.parquet', index=False)
df_lesoes.to_parquet('./raw_data/raw_injuries.parquet', index=False)
df_snaps.to_parquet('./raw_data/raw_snaps.parquet', index=False)
df_rosters.to_parquet('./raw_data/raw_rosters.parquet', index=False)

print("Processo finalizado. Prontos para tratamento.")

Exportando bases brutas para a próxima camada...
Processo finalizado. Prontos para tratamento.
